# Backtrader QuestDB strategy demo

Purpose: run research-only Backtrader strategy demos using OHLCV loaded from QuestDB `daily_prices`.

Scope constraints:
- read-only access to QuestDB;
- no OHLCV or FA ingestion;
- no DeepAgents integration in this notebook;
- generated CSV outputs stay under ignored `data/cache` paths.

## 1. Configure repo paths and demo scope

The notebook calls the checked-in helper scripts with `sys.executable`, so it uses the active Python environment on Windows.

In [ ]:
from pathlib import Path
import csv
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent

QUESTDB_URL = "http://localhost:9000"
SYMBOLS = "FPT,VNM,HPG"
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"
OUT_DIR = ROOT / "data" / "cache" / "backtrader"

def run_cmd(args):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}")
    return result

ROOT, OUT_DIR

## 2. Dependency check

Install with `pip install -r requirements-research.txt` if this cell reports that Backtrader is missing.

In [ ]:
import importlib.util
print("backtrader=", importlib.util.find_spec("backtrader"))

## 3. Export QuestDB data for Backtrader

The export helper reads from `daily_prices`, uses adjusted OHLC columns when present, validates OHLC/volume, and prints adjustment caveats.

In [ ]:
run_cmd([
    sys.executable,
    "scripts/export_questdb_ohlcv_for_backtrader.py",
    "--questdb-url", QUESTDB_URL,
    "--symbols", SYMBOLS,
    "--start-date", START_DATE,
    "--end-date", END_DATE,
    "--out-dir", str(OUT_DIR),
])

## 4. Data validation and adjusted-price caveat

Validation is performed by the export script before any Backtrader run:
- `high >= max(open, close)`
- `low <= min(open, close)`
- `volume >= 0`

Current caveat: adjusted OHLC is still source-unverified. If adjusted values equal raw values or `adjustment_status` includes `adjusted_price_missing_warn`, treat results as research-only.

## 5. Strategies

Implemented in `scripts/run_backtrader_questdb_demo.py`:
1. Buy and Hold baseline.
2. MA20/MA50 crossover.
3. RSI mean reversion.

In [ ]:
run_cmd([
    sys.executable,
    "scripts/run_backtrader_questdb_demo.py",
    "--questdb-url", QUESTDB_URL,
    "--symbols", SYMBOLS,
    "--start-date", START_DATE,
    "--end-date", END_DATE,
    "--out-dir", str(OUT_DIR),
])

## 6. Results table

The runner writes `backtrader_summary.csv` under `data/cache/backtrader`.

In [ ]:
summary_path = OUT_DIR / "backtrader_summary.csv"
with summary_path.open("r", newline="", encoding="utf-8") as fh:
    rows = list(csv.DictReader(fh))

for row in rows:
    print(
        row["symbol"],
        row["strategy"],
        "final=", f"{float(row['final_value']):,.0f}",
        "total%=", f"{float(row['total_return_pct']):.2f}",
        "ann%=", f"{float(row['annualized_return_pct']):.2f}",
        "max_dd%=", row["max_drawdown_pct"] or "n/a",
        "sharpe=", row["sharpe_ratio"] or "n/a",
    )

## 7. Equity curve plot

This cell plots if `pandas` and `matplotlib` are available. The notebook remains useful without them because the runner already prints and writes summary CSVs.

In [ ]:
equity_path = OUT_DIR / "backtrader_equity_curves.csv"
try:
    import pandas as pd
    import matplotlib.pyplot as plt

    equity = pd.read_csv(equity_path, parse_dates=["date"])
    ax = None
    for (symbol, strategy), frame in equity.groupby(["symbol", "strategy"]):
        label = f"{symbol} {strategy}"
        ax = frame.plot(x="date", y="value", label=label, ax=ax, figsize=(12, 6))
    ax.set_title("Backtrader equity curves")
    ax.set_ylabel("Portfolio value")
    plt.show()
except Exception as exc:
    print("plot skipped:", exc)
    print("equity CSV:", equity_path)

## 8. Why this is not wired into DeepAgents yet

Backtests are computational research tools, not general market-data answers. Before DeepAgents exposure, the system needs explicit routing and guardrails so backtests run only when the user asks for a backtest/simulation/strategy-performance task.

## 9. Next production steps

- Add QuestDB tables: `backtest_runs`, `backtest_metrics`, `backtest_trades`.
- Add a versioned strategy registry.
- Persist data snapshot metadata: symbol scope, date range, adjusted-price caveat, code commit.
- Add realistic costs, slippage, liquidity, and corporate-action validation.
- Add DeepAgents guardrails later: explicit backtest intent only, no real-money advice, and mandatory caveat disclosure.